# Signal Exploration

Use this notebook to answer: **does the signal I'm thinking about exist in the data?**

Workflow:
1. Load raw market data for a listing and time window
2. Compute signals and plot them alongside price
3. Measure signal properties (autocorrelation, distribution)
4. Check whether the signal predicts forward returns
5. Experiment with signal composition

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from gnomepy_research.explore import load_datastore, load_market_data, compute_signals, plot_signals
from gnomepy_research.signals import (
    MicropriceFairValue,
    MidFairValue,
    DepthImbalance,
    TradeImbalance,
    SpreadVolatility,
    BookPressure,
    MidMomentum,
    EWMA,
    Zscore,
    Weight,
)

## Configuration

Edit these cells to point at the data you want to explore.

In [ ]:
LISTING_ID = 1           # Change to the listing you want to explore
START = "2026-05-12T23:28:00"
END   = "2026-05-13T02:00:00"

## 1. Load Market Data

`load_market_data` returns a DataFrame indexed by event timestamp. It adds derived columns
(`mid_price`, `spread`, `spread_bps`, `microprice`) so you don't have to compute them.

In [ ]:
df = load_market_data(LISTING_ID, START, END)
print(f"{len(df):,} ticks loaded")
df.head()

In [ ]:
# Quick sanity checks
print("Time range:", df.index[0], "→", df.index[-1])
print("\nTop-of-book stats:")
df[["mid_price", "spread_bps", "bid_size_0", "ask_size_0"]].describe().round(4)

## 2. Compute Signals

`compute_signals` takes a `DataStore` (the raw binary source) and a dict of named signals.
It handles the update loop and scales FairValueSignal outputs to human-readable prices automatically.

In [ ]:
ds = load_datastore(LISTING_ID, START, END)

signals_df = compute_signals(ds, {
    "microprice":     MicropriceFairValue(),
    "depth_imbal":    DepthImbalance(num_levels=5, warmup=10),
    "trade_imbal":    TradeImbalance(window=50, warmup=20),
    "spread_vol":     SpreadVolatility(window=100, warmup=30),
})

print(f"Signal DataFrame: {signals_df.shape}")
signals_df.describe().round(4)

## 3. Plot Signals Alongside Price

In [ ]:
fig = plot_signals(
    df,
    signals_df,
    signals_to_plot=["depth_imbal", "trade_imbal"],
    title=f"Listing {LISTING_ID} — Depth & Trade Imbalance",
)
fig.show()

In [ ]:
# Overlay microprice vs mid on a single price panel
mid_ds = load_datastore(LISTING_ID, START, END)
fv_df = compute_signals(mid_ds, {
    "mid":        MidFairValue(),
    "microprice": MicropriceFairValue(),
})

fig = go.Figure()
for col in ["mid", "microprice"]:
    s = fv_df[col].dropna().iloc[::max(1, len(fv_df) // 5000)]
    fig.add_trace(go.Scatter(x=s.index, y=s.values, name=col, line=dict(width=0.8)))
fig.update_layout(title="Mid vs Microprice", height=400, hovermode="x unified")
fig.show()

## 4. Signal Statistics

Before trusting a signal, check: Is it mean-reverting or persistent? Is it normally distributed?
Does it have autocorrelation that could cause backtest lookahead?

In [ ]:
sig = signals_df["depth_imbal"].dropna()

print(f"Autocorrelation (lag 1):  {sig.autocorr(lag=1):.4f}")
print(f"Autocorrelation (lag 5):  {sig.autocorr(lag=5):.4f}")
print(f"Autocorrelation (lag 20): {sig.autocorr(lag=20):.4f}")
print()
print(f"Mean:   {sig.mean():.4f}")
print(f"Std:    {sig.std():.4f}")
print(f"Skew:   {sig.skew():.4f}")
print(f"Kurt:   {sig.kurt():.4f}")

In [ ]:
px.histogram(sig, nbins=100, title="Depth Imbalance Distribution").show()

## 5. Signal vs Forward Returns

The key question: does signal value at time T predict mid-price movement over the next N ticks?

In [ ]:
mid = df["mid_price"].dropna()

# Align signal and market data on timestamp
combined = pd.DataFrame({"signal": signals_df["depth_imbal"], "mid": mid})
combined = combined.dropna()

# Forward return at N ticks
def forward_return_corr(df, signal_col, n_ticks):
    fwd_ret = df["mid"].pct_change().shift(-n_ticks)
    return df[signal_col].corr(fwd_ret)

horizons = [1, 5, 10, 20, 50, 100]
corrs = {n: forward_return_corr(combined, "signal", n) for n in horizons}

corr_df = pd.Series(corrs, name="correlation")
corr_df.index.name = "forward_ticks"
print("Signal vs forward mid return:")
print(corr_df.round(4))

In [ ]:
# Scatter: signal value vs 10-tick forward return
n = 10
fwd = combined["mid"].pct_change().shift(-n) * 10_000  # in bps
plot_df = pd.DataFrame({"signal": combined["signal"], "fwd_bps": fwd}).dropna()
# Sample 5k points to keep it fast
sample = plot_df.sample(min(5000, len(plot_df)), random_state=42)
px.scatter(
    sample, x="signal", y="fwd_bps",
    title=f"Depth Imbalance vs {n}-tick forward return (bps)",
    opacity=0.3, trendline="ols",
).show()

## 6. Compare Signal Variants

Use `EWMA` and `Zscore` operations to smooth or normalize a signal.
Compare behavior across different parameterizations.

In [ ]:
ds2 = load_datastore(LISTING_ID, START, END)

base = DepthImbalance(num_levels=5, warmup=10)
variants_df = compute_signals(ds2, {
    "raw":            DepthImbalance(num_levels=5, warmup=10),
    "ewma_fast":      EWMA(DepthImbalance(num_levels=5, warmup=10), alpha=0.1, warmup=10),
    "ewma_slow":      EWMA(DepthImbalance(num_levels=5, warmup=10), alpha=0.01, warmup=100),
    "zscore":         Zscore(DepthImbalance(num_levels=5, warmup=10), horizon=200),
})

plot_signals(df, variants_df, title="Depth Imbalance Variants").show()

## 7. Signal Composition

Signals support arithmetic operators — you can combine them directly using `+`, `-`, `*`, `/`.
Use `Weight` to blend signals with explicit weights.

In [ ]:
ds3 = load_datastore(LISTING_ID, START, END)

depth = DepthImbalance(num_levels=5, warmup=10)
trade = TradeImbalance(window=50, warmup=20)
momentum = MidMomentum(fast_window=10, slow_window=50, warmup=50)

composed_df = compute_signals(ds3, {
    "depth":    DepthImbalance(num_levels=5, warmup=10),
    "trade":    TradeImbalance(window=50, warmup=20),
    "combined": Weight(
        [DepthImbalance(num_levels=5, warmup=10),
         TradeImbalance(window=50, warmup=20)],
        [0.6, 0.4],
    ),
})

plot_signals(df, composed_df, title="Composed Signal vs Components").show()